In [2]:
import pandas as pd                                                                   # pandas
from sklearn.model_selection import train_test_split                                  # train test split
from sklearn.compose import ColumnTransformer                                         # column transformer
from sklearn.preprocessing import OrdinalEncoder, KBinsDiscretizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import StandardScaler                                      # normalize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score) # metrics
from sklearn.metrics import classification_report                                     # classification report
from sklearn.metrics import ConfusionMatrixDisplay                                    # error matrix
import matplotlib.pyplot as plt                                                       # matplotlib
import numpy as np                                                                    # numpy
from lime.lime_tabular import LimeTabularExplainer                                    # lime

In [ ]:
df = pd.read_csv("dataset/ObesityDataSet_raw_and_data_sinthetic.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/ObesityDataSet_raw_and_data_sintetic.csv'

In [ ]:
# Let's check for missing values in the dataset
print(f"Missing values in the dataset: {df.isnull().sum().sum()}")

print("\n")

# Let's check for duplicate rows in the dataset
print(f"Duplicate rows in the dataset: {df.duplicated().sum()}")

# Let's remove duplicate rows
if(df.duplicated().sum() > 0):
   df = df.drop_duplicates()

In [ ]:
x = df.drop("NObeyesdad", axis=1) # axis=1 == axis='columns'

y = df["NObeyesdad"]

In [ ]:
# Let's check the distribution of the target

print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

In [ ]:
# Let's separate the features into categorical and numerical

categorical_features = [
                "Gender", 
                "family_history_with_overweight", 
                "FAVC",
                "CAEC",
                "SMOKE",
                "SCC",
                "CALC",
                "MTRANS"
                    ]

numeric_features = [
                "Age",
                "Height",
                "Weight",
                "FCVC",
                "NCP",
                "CH2O",
                "FAF",
                "TUE"
                    ]

In [ ]:
# Let's split the dataset into training and testing

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1, stratify=y) 

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("numerical", KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile"), numeric_features),
    ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features)
])

In [ ]:
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

n_numeric = len(numeric_features)

x_train_processed[:, n_numeric:] += 1
x_test_processed[:, n_numeric:] += 1

x_train_processed = x_train_processed.astype(int)
x_test_processed = x_test_processed.astype(int)

In [ ]:
nb = CategoricalNB()

In [ ]:
nb.fit(x_train_processed, y_train)

In [ ]:
pred_nb = nb.predict(x_test_processed)

In [ ]:
# Avaliar
metrics_df = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, pred_nb),
    "Precision": precision_score(
        y_test, pred_nb, average="macro", zero_division=0
    ),
    "Recall": recall_score(
        y_test, pred_nb, average="macro", zero_division=0
    ),
    "F1-score": f1_score(
        y_test, pred_nb, average="macro", zero_division=0
    )
}], index=["CategoricalNB"])

print(metrics_df.round(3))
print(classification_report(y_test, pred_nb, zero_division=0))

# Matriz de confusão
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_nb,
    xticks_rotation=90,
    ax=ax,
    colorbar=False
)
ax.set_title("Categorical Naive Bayes")
plt.tight_layout()
plt.show()